# Validaciones y Exploración

Valida la calidad de `ANALYTICS.OBT_TRIPS`:
- Nulos en campos esenciales
- Rangos logicos (distancia, duracion, montos)
- Coherencia de fechas (pickup <= dropoff)
- Conteos por mes/servicio

## Imports y configuración

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

SNOWFLAKE_ACCOUNT   = os.environ["SNOWFLAKE_ACCOUNT"]
SNOWFLAKE_USER      = os.environ["SNOWFLAKE_USER"]
SNOWFLAKE_PASSWORD  = os.environ["SNOWFLAKE_PASSWORD"]
SNOWFLAKE_DATABASE  = os.environ["SNOWFLAKE_DATABASE"]
SNOWFLAKE_WAREHOUSE = os.environ["SNOWFLAKE_WAREHOUSE"]
SNOWFLAKE_ROLE      = os.environ.get("SNOWFLAKE_ROLE", "SYSADMIN")
SCHEMA_ANALYTICS    = os.environ.get("SNOWFLAKE_SCHEMA_ANALYTICS", "ANALYTICS")

spark = SparkSession.builder \
    .appName("P3_Validaciones") \
    .config("spark.jars.packages",
            "net.snowflake:spark-snowflake_2.12:2.16.0-spark_3.4,"
            "net.snowflake:snowflake-jdbc:3.16.1") \
    .getOrCreate()

sf_options = {
    "sfURL": f"{SNOWFLAKE_ACCOUNT}.snowflakecomputing.com",
    "sfUser": SNOWFLAKE_USER, "sfPassword": SNOWFLAKE_PASSWORD,
    "sfDatabase": SNOWFLAKE_DATABASE, "sfSchema": SCHEMA_ANALYTICS,
    "sfWarehouse": SNOWFLAKE_WAREHOUSE, "sfRole": SNOWFLAKE_ROLE,
}

print(f"Spark {spark.version} | Validando {SNOWFLAKE_DATABASE}.{SCHEMA_ANALYTICS}.OBT_TRIPS")

## Cargar OBT completa

In [ ]:
df = spark.read.format("net.snowflake.spark.snowflake") \
    .options(**sf_options) \
    .option("dbtable", "OBT_TRIPS") \
    .load()

df = df.toDF(*[c.lower() for c in df.columns])
total_rows = df.count()
print(f"OBT_TRIPS: {total_rows:,} filas | {len(df.columns)} columnas")

## Validación de nulos en campos esenciales

In [ ]:
essential_cols = [
    "pickup_datetime", "dropoff_datetime", "pu_location_id",
    "do_location_id", "service_type", "trip_distance", "total_amount"
]

print(f"{'Columna':<25} {'Nulos':>12} {'% del total':>12}")
print("-" * 50)
for col in essential_cols:
    if col in df.columns:
        null_count = df.filter(F.col(col).isNull()).count()
        pct = (null_count / total_rows * 100) if total_rows > 0 else 0
        flag = " !!" if pct > 5 else ""
        print(f"{col:<25} {null_count:>12,} {pct:>11.2f}%{flag}")
    else:
        print(f"{col:<25} {'NO EXISTE':>12}")

## Validación de rangos logicos

In [ ]:
range_checks = [
    ("trip_distance < 0", df.filter(F.col("trip_distance") < 0).count()),
    ("trip_duration_min < 0", df.filter(F.col("trip_duration_min") < 0).count()),
    ("total_amount < 0", df.filter(F.col("total_amount") < 0).count()),
    ("fare_amount < 0", df.filter(F.col("fare_amount") < 0).count()),
    ("tip_amount < 0", df.filter(F.col("tip_amount") < 0).count()),
    ("avg_speed_mph > 200", df.filter(F.col("avg_speed_mph") > 200).count()),
    ("trip_duration_min > 1440 (>24h)", df.filter(F.col("trip_duration_min") > 1440).count()),
]

print(f"{'Regla':<35} {'Violaciones':>12} {'% del total':>12}")
print("-" * 60)
for rule, count in range_checks:
    pct = (count / total_rows * 100) if total_rows > 0 else 0
    flag = " !!" if pct > 1 else ""
    print(f"{rule:<35} {count:>12,} {pct:>11.2f}%{flag}")

## Coherencia de fechas

In [ ]:
bad_dates = df.filter(F.col("pickup_datetime") > F.col("dropoff_datetime")).count()
pct = (bad_dates / total_rows * 100) if total_rows > 0 else 0
print(f"Filas con pickup > dropoff: {bad_dates:,} ({pct:.2f}%)")

# Fechas fuera de rango esperado (antes de 2015 o en el futuro)
out_of_range = df.filter(
    (F.year("pickup_datetime") < 2009) | (F.year("pickup_datetime") > 2026)
).count()
print(f"Filas con pickup_year fuera [2009-2026]: {out_of_range:,}")

## Conteos por servicio y mes

In [ ]:
counts = df.groupBy("service_type", "source_year", "source_month") \
    .count() \
    .orderBy("service_type", "source_year", "source_month")

counts.show(300, truncate=False)

## Estadisticas descriptivas de columnas numericas

In [ ]:
numeric_cols = [
    "trip_distance", "fare_amount", "total_amount",
    "tip_amount", "trip_duration_min", "avg_speed_mph", "tip_pct"
]

df.select(numeric_cols).describe().show(truncate=False)

## Resumen de validación

In [ ]:
services = df.select("service_type").distinct().count()
years = df.select("source_year").distinct().count()
months_covered = df.select("source_year", "source_month").distinct().count()

print("=" * 50)
print("RESUMEN DE VALIDACION")
print("=" * 50)
print(f"Total filas OBT:       {total_rows:,}")
print(f"Servicios:             {services}")
print(f"Anios cubiertos:       {years}")
print(f"Meses con datos:       {months_covered}")
print(f"Columnas:              {len(df.columns)}")
print(f"Pickup > Dropoff:      {bad_dates:,}")
print(f"Fechas fuera de rango: {out_of_range:,}")